In [0]:
%sql
DESCRIBE DETAIL financial_project_db.silver_crypto_history;

### HISTORY TABLE was not partitioned
- delta lake does not support alter table partition by
- so recreate the data

In [0]:
df = spark.table("financial_project_db.silver_crypto_history")

In [0]:
from common_scripts.common_functions import * 

setup_adls_access(spark, dbutils)



In [0]:
df.write.format("delta")\
    .mode("overwrite")\
    .partitionBy("data_load_date")\
    .save("abfss://financial-data-project@financialdatalakestorage.dfs.core.windows.net/silver/crypto_history_partitioned")

In [0]:
%sql
DROP TABLE financial_project_db.silver_crypto_history;

CREATE TABLE financial_project_db.silver_crypto_history
USING DELTA
LOCATION 'abfss://financial-data-project@financialdatalakestorage.dfs.core.windows.net/silver/crypto_history_partitioned';

- Optimize Combines small files into bigger ones
- Improves scan performance

In [0]:
%sql
OPTIMIZE financial_project_db.silver_crypto_history;

-to improve filtering speed and Join performance ZORDER by id (since queries run based on ID)

In [0]:
%sql
OPTIMIZE financial_project_db.silver_crypto_history
ZORDER BY (id);

In [0]:
%sql
VACUUM financial_project_db.silver_crypto_history RETAIN 168 HOURS;

## Simulating small file problem 

In [0]:
silver_path = "abfss://financial-data-project@financialdatalakestorage.dfs.core.windows.net/silver/crypto_history_partitioned"
df = spark.read.format("delta").load(silver_path)

df.repartition(100) \
  .write \
  .format("delta") \
  .mode("overwrite") \
  .save(silver_path)

In [0]:
%sql
DESCRIBE DETAIL financial_project_db.silver_crypto_history;